# Step 8: the holdout

The plan was committed in `87aa8d5` before any of this data was looked at. Each test runs once. Nothing changes after looking.

- **Test years:** 2023 (from 17 Mar), 2024, 2025, 2026 (to 23 Oct).
- **Each year:** train up to Y-2, tune on Y-1, test on Y.
- **Direction tests:** scored against the fair bar with McNemar's test and AUC.

Each finished run is saved in `results/holdout_*`. **Don't delete those files and rerun with changes:** that would no longer be a one-shot test.

In [1]:
from functools import partial
from pathlib import Path

import pandas as pd
from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression

from classifiers import CLASSIFIERS, HIDDEN_GRID, fit_classifier, fit_net, model_table, overview, pick_net_setting
from data import HOLDOUT_START, load_all, load_us
from evaluation import HOLDOUT_YEARS, mcnemar, money_test, paired_loss_test
from features import make_har, make_har_with_us, make_indicators, make_range_sequence, make_us_features, up_close_to_close, up_open_to_close
from gru import HIDDEN_SIZES
from models import pooled, report, walk_forward
from volatility import fit_gru_vol, fit_regressor, pick_gru_vol_setting, qlike, walk_forward_garch, walk_forward_vol

markets = load_all(include_holdout=True)
us = load_us(include_holdout=True)
us_feature = partial(make_us_features, us=us)
RESULTS = Path('results')
RESULTS.mkdir(exist_ok=True)
pd.set_option('display.max_colwidth', None)

def run_market(path, run, df):
    """Load the saved holdout predictions, or run the model once and save them."""
    if path.exists():
        return pd.read_pickle(path)
    preds = run(df)
    preds = preds[preds.index >= HOLDOUT_START]  # 2023 is scored from the holdout start
    preds.to_pickle(path)
    return preds

def run_markets(name, run, names=None):
    names = names or list(markets)
    runs = Parallel(n_jobs=2)(delayed(run_market)(RESULTS / f'holdout_{name}_{m}.pkl', run, markets[m]) for m in names)
    return dict(zip(names, runs))

def direction(fit, grid, make_x, make_y=up_close_to_close, **options):
    return partial(walk_forward, fit=fit, grid=grid, make_x=make_x, make_y=make_y, test_years=HOLDOUT_YEARS, **options)

def vol(fit, settings, make_x, **options):
    return partial(walk_forward_vol, fit=fit, settings=settings, make_x=make_x, test_years=HOLDOUT_YEARS, **options)

## Data check

Days in the holdout, and the share of fake opens (in %) by year. Fake-open days are left out of the open-to-close target.

In [2]:
rows = {}
for market, df in markets.items():
    h = df[df.index >= HOLDOUT_START]
    fake = h['Open'] == h['Close'].shift(1)
    rows[market] = {'days': len(h), 'first': h.index[0].date(), 'last': h.index[-1].date(),
                    **(100 * fake.groupby(h.index.year).mean()).round(0).to_dict()}
pd.DataFrame(rows).T

,days,first,last,2023,2024,2025,2026
Indonesia,852,2023-03-17,2026-10-23,56.0,0.0,0.0,0.0
Malaysia,884,2023-03-17,2026-10-23,0.0,0.0,0.0,0.0
Philippines,879,2023-03-17,2026-10-23,0.0,1.0,0.0,0.0
Singapore,906,2023-03-17,2026-10-23,0.0,0.0,0.0,0.0
Thailand,872,2023-03-17,2026-10-22,0.0,0.0,0.0,0.0
India,886,2023-03-17,2026-10-23,0.0,0.0,0.0,0.0


## Test 1: the paper's setup

All seven models on the 12 indicators, close-to-close. **Expected: no edge.**

In [3]:
test1 = {name: run_markets(f'paper_{name}', direction(fit_classifier, models, make_indicators))
         for name, models in CLASSIFIERS.items()}

In [4]:
test1['Neural network'] = run_markets('paper_Neural network', direction(fit_net, HIDDEN_GRID, make_indicators, pick=pick_net_setting))

In [5]:
display(overview(test1).round(3))
pd.DataFrame({name: {market: pooled(p)['accuracy %'] - pooled(p)['fair bar %'] for market, p in by_market.items()}
              for name, by_market in test1.items()}).T.round(1)

,mean AUC,above fair bar,below fair bar,significantly above,significantly below,above stretch bar,one-answer folds
LDA,0.510,7.0,16.0,0.0,3.0,3.0,0.0
Logistic regression,0.515,5.0,11.0,0.0,1.0,4.0,5.0
KNN,0.510,9.0,14.0,0.0,1.0,3.0,1.0
SVM,0.509,6.0,10.0,0.0,1.0,2.0,5.0
Random forest,0.513,10.0,11.0,1.0,1.0,7.0,0.0
AdaBoost,0.516,12.0,11.0,0.0,1.0,7.0,0.0
Neural network,0.509,8.0,15.0,0.0,0.0,3.0,0.0


,Indonesia,Malaysia,Philippines,Singapore,Thailand,India
LDA,-1.8,1.4,1.4,-3.6,-1.6,-1.6
Logistic regression,-0.8,2.0,1.5,-3.6,-0.8,0.2
KNN,-0.5,0.7,0.2,-2.6,-0.9,-0.1
SVM,0.0,-0.5,0.3,-3.1,0.2,0.3
Random forest,-1.3,3.2,2.4,-4.2,-2.6,1.8
AdaBoost,-4.7,1.8,0.7,-2.4,-1.1,-0.2
Neural network,-0.7,0.9,1.4,-2.5,-0.1,0.6


In [6]:
for name, by_market in test1.items():
    print(name)
    display(model_table(by_market).round(3))

LDA


,accuracy %,MCC,AUC,fair bar %,p vs fair,same as yesterday %,p vs same as yesterday,called up %,one-answer folds
Indonesia,50.469,-0.045,0.466,52.230,0.124,49.648,0.772,90.258,0.0
Malaysia,51.697,0.033,0.523,50.339,0.539,51.018,0.786,63.801,0.0
Philippines,51.650,0.035,0.546,50.284,0.452,48.350,0.158,75.654,0.0
Singapore,50.773,-0.008,0.497,54.415,0.079,53.532,0.259,63.245,0.0
Thailand,48.853,-0.036,0.491,50.459,0.322,51.720,0.164,80.275,0.0
India,52.144,0.008,0.543,53.725,0.393,52.935,0.674,73.815,0.0


Logistic regression


,accuracy %,MCC,AUC,fair bar %,p vs fair,same as yesterday %,p vs same as yesterday,called up %,one-answer folds
Indonesia,51.408,-0.068,0.485,52.230,0.065,49.648,0.493,98.709,3.0
Malaysia,52.376,0.050,0.533,50.339,0.272,51.018,0.565,72.851,1.0
Philippines,51.763,0.040,0.543,50.284,0.353,48.350,0.148,81.001,0.0
Singapore,50.773,-0.022,0.494,54.415,0.051,53.532,0.232,70.309,0.0
Thailand,49.656,-0.021,0.501,50.459,0.558,51.720,0.351,87.959,1.0
India,53.950,0.043,0.540,53.725,0.940,52.935,0.608,79.910,0.0


KNN


,accuracy %,MCC,AUC,fair bar %,p vs fair,same as yesterday %,p vs same as yesterday,called up %,one-answer folds
Indonesia,51.761,-0.006,0.482,52.230,0.704,49.648,0.418,92.723,1.0
Malaysia,51.018,0.021,0.516,50.339,0.815,51.018,1.000,48.643,0.0
Philippines,50.512,0.009,0.515,50.284,0.952,48.350,0.327,68.601,0.0
Singapore,51.766,-0.000,0.493,54.415,0.161,53.532,0.485,70.199,0.0
Thailand,49.541,-0.013,0.510,50.459,0.687,51.720,0.291,65.367,0.0
India,53.612,0.036,0.549,53.725,1.000,52.935,0.754,78.894,0.0


... (4 outputs left out)

AdaBoost


,accuracy %,MCC,AUC,fair bar %,p vs fair,same as yesterday %,p vs same as yesterday,called up %,one-answer folds
Indonesia,47.535,-0.063,0.486,52.230,0.029,49.648,0.441,62.676,0.0
Malaysia,52.149,0.043,0.511,50.339,0.405,51.018,0.658,63.348,0.0
Philippines,50.967,0.019,0.532,50.284,0.763,48.350,0.260,68.828,0.0
Singapore,51.987,0.011,0.492,54.415,0.225,53.532,0.568,66.887,0.0
Thailand,49.312,-0.014,0.481,50.459,0.660,51.720,0.300,52.064,0.0
India,53.499,0.039,0.535,53.725,0.947,52.935,0.838,74.492,0.0


Neural network


,accuracy %,MCC,AUC,fair bar %,p vs fair,same as yesterday %,p vs same as yesterday,called up %,one-answer folds
Indonesia,51.526,-0.013,0.471,52.230,0.539,49.648,0.484,92.254,0.0
Malaysia,51.244,0.026,0.517,50.339,0.750,51.018,0.959,45.475,0.0
Philippines,51.650,0.033,0.542,50.284,0.528,48.350,0.130,65.415,0.0
Singapore,51.876,-0.012,0.480,54.415,0.126,53.532,0.476,77.152,0.0
Thailand,50.344,0.007,0.508,50.459,1.000,51.720,0.487,47.592,0.0
India,54.289,0.060,0.551,53.725,0.802,52.935,0.464,71.445,0.0


## Test 2: close-to-close with the US feature

Logistic regression, all six markets. **Expected: beats the fair bar, though it can't be traded.**

In [7]:
test2 = run_markets('us_close', direction(fit_classifier, CLASSIFIERS['Logistic regression'], us_feature))
model_table(test2).round(3)

,accuracy %,MCC,AUC,fair bar %,p vs fair,same as yesterday %,p vs same as yesterday,called up %,one-answer folds
Indonesia,56.690,0.131,0.584,52.230,0.008,49.648,0.003,77.230,0.0
Malaysia,57.692,0.154,0.623,50.339,0.001,51.018,0.005,56.900,0.0
Philippines,54.949,0.103,0.585,50.284,0.021,48.350,0.005,65.984,0.0
Singapore,57.726,0.143,0.622,54.415,0.138,53.532,0.065,57.837,0.0
Thailand,53.096,0.067,0.556,50.459,0.130,51.720,0.591,75.803,0.0
India,56.998,0.119,0.593,53.725,0.049,52.935,0.076,77.088,0.0


In [8]:
pd.concat({market: report(p)[['accuracy %', 'fair bar %', 'p vs fair', 'AUC']] for market, p in test2.items()}).round(3)

accuracy %  fair bar %  p vs fair    AUC
            year                                          
Indonesia   2023      60.215      56.452      0.349  0.600
            2024      54.852      51.055      0.253  0.573
            2025      55.932      55.932      1.000  0.567
            2026      56.477      45.078      0.003  0.609
Malaysia    2023      62.051      46.154      0.001  0.704
            2024      55.466      53.441      0.702  0.596
            2025      55.328      50.820      0.315  0.623
            2026      59.091      50.000      0.082  0.594
Philippines 2023      59.474      51.053      0.072  0.645
            2024      52.245      50.204      0.668  0.574
            2025      51.440      50.617      0.912  0.550
            2026      58.209      49.254      0.027  0.631
Singapore   2023      56.566      50.505      0.213  0.647
            2024      54.365      55.159      0.924  0.619
            2025      59.921      55.952      0.373  0.614
            2026      60.294      55.392      0.348  0.615
Thailand    2023      51.309      50.262      0.883  0.565
            2024      50.000      47.131      0.450  0.553
            2025      53.719      50.000      0.342  0.552
            2026      57.949      55.385      0.473  0.595
India       2023      65.803      62.694      0.497  0.655
            2024      57.959      55.102      0.427  0.596
            2025      53.414      50.201      0.396  0.548
            2026      51.759      47.739      0.152  0.627

## Test 3: open-to-close with the US feature

Logistic regression for the Philippines, random forest for Malaysia, plus the money test.
**Expected: any edge stays below trading costs.** These assume you could buy at the index's opening number, which may not be a real price.

In [9]:
FOREST_US = [RandomForestClassifier(n_estimators=300, min_samples_leaf=leaf, random_state=0, n_jobs=-1) for leaf in [10, 50, 200]]

test3 = {
    **run_markets('open_lr', direction(fit_classifier, CLASSIFIERS['Logistic regression'], us_feature, make_y=up_open_to_close), ['Philippines']),
    **run_markets('open_forest', direction(fit_classifier, FOREST_US, us_feature, make_y=up_open_to_close), ['Malaysia']),
}
display(model_table(test3).round(3))
pd.concat({market: report(p)[['setting', 'accuracy %', 'fair bar %', 'p vs fair', 'AUC']] for market, p in test3.items()}).round(3)

,accuracy %,MCC,AUC,fair bar %,p vs fair,same as yesterday %,p vs same as yesterday,called up %,one-answer folds
Philippines,53.425,0.084,0.559,48.973,0.044,50.114,0.171,57.420,0.0
Malaysia,51.642,0.033,0.539,50.057,0.528,53.681,0.411,51.982,0.0


setting  \
            year                                                                                              
Philippines 2023                                                                   LogisticRegression(C=10)   
            2024                                                                   LogisticRegression(C=10)   
            2025                                                                   LogisticRegression(C=10)   
            2026                                                                LogisticRegression(C=0.001)   
Malaysia    2023  RandomForestClassifier(min_samples_leaf=200, n_estimators=300, n_jobs=-1, random_state=0)   
            2024  RandomForestClassifier(min_samples_leaf=200, n_estimators=300, n_jobs=-1, random_state=0)   
            2025  RandomForestClassifier(min_samples_leaf=200, n_estimators=300, n_jobs=-1, random_state=0)   
            2026   RandomForestClassifier(min_samples_leaf=50, n_estimators=300, n_jobs=-1, random_state=0)   

                  accuracy %  fair bar %  p vs fair    AUC  
            year                                            
Philippines 2023      58.421      50.526      0.101  0.595  
            2024      49.794      45.267      0.294  0.563  
            2025      50.206      44.033      0.163  0.539  
            2026      57.000      58.000      0.917  0.565  
Malaysia    2023      55.897      52.308      0.515  0.611  
            2024      49.187      52.846      0.460  0.514  
            2025      52.049      48.770      0.523  0.550  
            2026      50.000      45.960      0.488  0.505

In [10]:
COSTS = [0.001, 0.0025]  # round trip: 0.1% and 0.25%

rows = {}
for market, p in test3.items():
    df = markets[market].loc[p.index]
    day_return = df['Close'] / df['Open'] - 1
    calls_up = p['p_up'] > 0.5
    row = {'days': len(p), 'days in': calls_up.sum(), 'average day in %': 100 * day_return[calls_up].mean()}
    for cost in COSTS:
        model, every_day = money_test(day_return, calls_up, cost)
        row[f'model % (cost {cost:.2%})'] = 100 * (model - 1)
        row[f'every day % (cost {cost:.2%})'] = 100 * (every_day - 1)
    rows[market] = row
pd.DataFrame(rows).round(2)

,Philippines,Malaysia
days,876.00,883.00
days in,503.00,459.00
average day in %,-0.03,0.03
model % (cost 0.10%),-47.80,-27.87
every day % (cost 0.10%),-82.66,-61.63
model % (cost 0.25%),-75.49,-63.80
every day % (cost 0.25%),-95.36,-89.82


## Test 4: India, same as yesterday vs the fair bar

Close-to-close. Is India's direction persistence real?

In [11]:
p = test2['India']
only_same, only_fair, p_value = mcnemar(p['up'], p['same_as_yesterday'], p['fair_call'])
pd.Series({
    'days': len(p),
    'same as yesterday %': 100 * (p['same_as_yesterday'] == p['up']).mean(),
    'fair bar %': 100 * (p['fair_call'] == p['up']).mean(),
    'days only same-as-yesterday got right': only_same,
    'days only the fair bar got right': only_fair,
    'p': p_value,
}).round(3)

days                                     886.000
same as yesterday %                       52.935
fair bar %                                53.725
days only same-as-yesterday got right    201.000
days only the fair bar got right         208.000
p                                          0.767
dtype: float64

## Test 5: volatility

HAR vs same as yesterday, GARCH, the GRU (seeds 0, 1, 2) and HAR + US, on R2 and QLIKE, with the paired test on both.
**Expected: HAR beats the first three on R2, and HAR + US adds a little.**

In [12]:
LINE = [LinearRegression()]
har = run_markets('har', vol(fit_regressor, LINE, make_har))
har_us = run_markets('har_us', vol(fit_regressor, LINE, partial(make_har_with_us, us=us)))
garch = run_markets('garch', partial(walk_forward_garch, test_years=HOLDOUT_YEARS))

In [13]:
gru = {seed: run_markets(f'gru_seed{seed}', vol(partial(fit_gru_vol, seed=seed), HIDDEN_SIZES, make_range_sequence,
                                                  pick=partial(pick_gru_vol_setting, seed=seed)))
       for seed in [0, 1, 2]}

In [14]:
same = {m: p.assign(predicted=p['same_as_yesterday']) for m, p in har.items()}
MODELS = {'same as yesterday': same, 'HAR': har, 'HAR + US': har_us, 'GARCH': garch,
          'GRU 0': gru[0], 'GRU 1': gru[1], 'GRU 2': gru[2]}

for runs in MODELS.values():  # every model must be scored on the same days
    for m in markets:
        assert runs[m].index.equals(har[m].index), m

def r2(p):
    error = p['predicted'] - p['actual']
    error_of_average = p['average_so_far'] - p['actual']
    return 1 - (error ** 2).sum() / (error_of_average ** 2).sum()

def squared_error(p):
    return (p['predicted'] - p['actual']) ** 2

def table(score):
    return pd.DataFrame({name: {m: score(runs[m], m) for m in markets} for name, runs in MODELS.items()}).round(3)

def vs_har(loss):
    return table(lambda p, m: float('nan') if p is har[m] else paired_loss_test(loss(p), loss(har[m])))

for title, t in {
    'R2': table(lambda p, m: r2(p)),
    'p vs HAR on squared errors': vs_har(squared_error),
    'QLIKE': table(lambda p, m: qlike(p).mean()),
    'p vs HAR on QLIKE': vs_har(qlike),
}.items():
    print(title)
    display(t)

R2


,same as yesterday,HAR,HAR + US,GARCH,GRU 0,GRU 1,GRU 2
Indonesia,0.222,0.471,0.462,0.367,0.462,0.474,0.468
Malaysia,-0.345,0.115,0.113,0.040,0.094,0.103,0.116
Philippines,-0.590,0.058,0.063,0.029,0.053,0.054,0.060
Singapore,-0.251,0.190,0.195,0.137,0.170,0.184,0.184
Thailand,-0.288,0.244,0.244,0.191,0.190,0.194,0.213
India,-0.109,0.302,0.301,0.228,0.285,0.300,0.296


p vs HAR on squared errors


,same as yesterday,HAR,HAR + US,GARCH,GRU 0,GRU 1,GRU 2
Indonesia,0.0,NaN,0.029,0.000,0.372,0.663,0.572
Malaysia,0.0,NaN,0.747,0.022,0.165,0.117,0.898
Philippines,0.0,NaN,0.598,0.117,0.656,0.732,0.897
Singapore,0.0,NaN,0.522,0.044,0.149,0.655,0.608
Thailand,0.0,NaN,0.995,0.017,0.008,0.016,0.078
India,0.0,NaN,0.640,0.014,0.062,0.705,0.313


QLIKE


,same as yesterday,HAR,HAR + US,GARCH,GRU 0,GRU 1,GRU 2
Indonesia,0.626,0.487,0.501,0.670,0.509,0.483,0.495
Malaysia,0.715,0.460,0.455,0.498,0.504,0.475,0.468
Philippines,0.802,0.420,0.416,0.464,0.436,0.436,0.430
Singapore,0.493,0.327,0.317,0.351,0.314,0.315,0.320
Thailand,0.557,0.321,0.322,0.366,0.370,0.373,0.357
India,0.840,0.489,0.489,0.437,0.489,0.487,0.485


p vs HAR on QLIKE


,same as yesterday,HAR,HAR + US,GARCH,GRU 0,GRU 1,GRU 2
Indonesia,0.001,NaN,0.065,0.000,0.250,0.751,0.253
Malaysia,0.000,NaN,0.431,0.040,0.034,0.015,0.218
Philippines,0.000,NaN,0.444,0.003,0.112,0.169,0.260
Singapore,0.000,NaN,0.132,0.181,0.221,0.310,0.413
Thailand,0.000,NaN,0.550,0.001,0.001,0.000,0.004
India,0.000,NaN,0.777,0.212,0.994,0.735,0.508


HAR's R2 in each test year.

In [15]:
pd.DataFrame({m: {year: r2(f) for year, f in p.groupby('year')} for m, p in har.items()}).T.round(2)

,2023,2024,2025,2026
Indonesia,0.22,0.12,0.34,0.67
Malaysia,0.18,0.07,0.13,0.09
Philippines,0.06,-0.00,0.14,0.02
Singapore,-0.04,0.14,0.31,0.18
Thailand,0.02,-0.02,0.42,0.33
India,0.55,0.08,0.31,0.29


Not a planned test, decided before looking: Indonesia's 2023 has many fake opens, so HAR's R2 with those days left out.

In [16]:
p = har['Indonesia']
df = markets['Indonesia']
fake = (df['Open'] == df['Close'].shift(1))[p.index]
pd.Series({'all days': r2(p), 'real opens': r2(p[~fake]), 'fake-open days': int(fake.sum())}).round(3)

all days            0.471
real opens          0.484
fake-open days    107.000
dtype: float64